In [30]:
from typing import List
from enum import Enum, auto
import random
import networkx as nx #Python library for working with graphs
from collections import deque #deque, Double Ended Queue
from itertools import product #product is used to generate all possible True/False combinations for the parent variables.
!pip install pomegranate
from pomegranate.bayesian_network import BayesianNetwork
from pomegranate.distributions import Categorical, ConditionalCategorical #a simple prior distribution. #a conditional distribution that depends on parent variables
random.seed(0)

In [31]:
# Make percept a clean data container that the Environment can create and the Agent can read.
class Percept():
  time_step: int
  bump : bool
  breeze : bool
  stench : bool
  scream : bool
  glitter : bool
  reward : int
  done : bool

  def __init__(self, time_step: int, bump : bool, breeze : bool, stench : bool, scream : bool, glitter : bool, reward : int, done : bool):
    self.time_step = time_step
    self.bump = bump
    self.breeze = breeze
    self.stench = stench
    self.scream = scream
    self.glitter = glitter
    self.reward = reward
    self.done = done

  def __str__(self):
    return ( f"t={self.time_step} | "
     f"bump={self.bump}, breeze={self.breeze}, stench={self.stench}, scream={self.scream}, glitter={self.glitter} | "
     f"reward={self.reward} | "
     f"done={self.done}"
    )

In [32]:
# let's test the class
#perectTest=Percept(1,True,False,False,False,False,-11,False)
#print(perectTest)
#print(perectTest.bump)
#perectTest2=Percept(1,True,False,False,False,False,-11,True)
#print(perectTest2)

In [33]:
class Action(Enum):
  LEFT = auto()
  RIGHT = auto()
  FORWARD = auto()
  GRAB = auto()
  SHOOT = auto()
  CLIMB =  auto()

In [34]:
# Let's test Action
#list(Action)
#random.choice(list(Action))

In [35]:
class Orientation(Enum):
  E = auto()
  S = auto()
  W = auto()
  N = auto()

  def symbol(self)-> str:
    return self.name

  def turn_right(self)->'Orientation':
    if self == Orientation.E:
      return Orientation.S
    if self == Orientation.S:
      return Orientation.W
    if self == Orientation.W:
      return Orientation.N
    else:
      return Orientation.E

  def turn_left(self)->'Orientation':
    if self == Orientation.E:
      return Orientation.N
    if self == Orientation.N:
      return Orientation.W
    if self == Orientation.W:
      return Orientation.S
    else:
      return Orientation.E

In [36]:
# Let's test orientation
# Orientation.N.symbol()
# print(Orientation.E.turn_right())
# print(Orientation.E.turn_left())

In [37]:
class NaiveAgent():
  def choose_action(self):
    return random.choice(list(Action))

  def run(self):
    env = Environment() # this is creating an object of env from Environment Class
    cumulative_reward = 0
    percept = env.init(0.2, False)
    while not percept.done:
      env.visualize()
      print("Percept:", percept)
      action = self.choose_action()
      print("Action:", action)
      percept = env.step(action)
      cumulative_reward += percept.reward
    env.visualize()
    print("Percept:", percept)
    print('Cumulative reward:', cumulative_reward)

In [38]:
# Let's test NaiveAgent class
# n = NaiveAgent()
# print(n.choose_action())

In [39]:
class Location():
  x: int
  y: int

  def __init__(self, x: int, y: int):
    self.x = x
    self.y = y

  def __str__(self):
    return f'({self.x},{self.y})'

  def is_left_of(self, location: 'Location')-> bool:
    return self.x == location.x - 1 and self.y == location.y

  def is_right_of(self, location: 'Location')->bool:
    return self.x == location.x + 1 and self.y == location.y

  def is_above(self, location: 'Location')->bool:
    return self.x == location.x and self.y == location.y + 1

  def is_below(self, location: 'Location')->bool:
    return self.x == location.x and self.y == location.y - 1

  def is_location(self, location: 'Location')->bool:
    return self.x == location.x and self.y == location.y

  def at_left_edge(self)->bool:
    return self.x == 0

  def at_right_edge(self)->bool:
    return self.x == 3

  def at_bottom_edge(self)->bool:
    return self.y == 0

  def at_top_edge(self)->bool:
    return self.y == 3

  def neighbours(self)->list('location'):
    neighbours = []
    if not self.at_left_edge():
      neighbours.append(Location(self.x-1, self.y))
    if not self.at_right_edge():
      neighbours.append(Location(self.x+1, self.y))
    if not self.at_bottom_edge():
      neighbours.append(Location(self.x, self.y-1))
    if not self.at_top_edge():
      neighbours.append(Location(self.x, self.y+1))
    return neighbours
  def set_to(self, location:'Location'):
    self.x = location.x
    self.y = location.y
  def forward(self, orientation)->bool: #return True if bumped a wall
    if orientation == Orientation.E:
      if self.at_right_edge():
        return True
      self.x +=1
      return False

    if orientation == Orientation.S:
      if self.at_bottom_edge():
        return True
      self.y -=1
      return False

    if orientation == Orientation.W:
      if self.at_left_edge():
        return True
      self.x -=1
      return False

    # orientation == Orientation.N:
    if self.at_top_edge():
      return True
    self.y +=1
    return False

  @staticmethod
  def from_linear(n:int)->'location':
    x = n % 4
    y = n // 4
    return Location(x,y)

  def to_linear(self)->int:
    return self.x + self.y * 4

  @staticmethod
  def random()->'location':
    return Location(random.randint(0,3),random.randint(0,3))

In [40]:
# Let's test Location class
#print(Location(1,2))
#print(Location(1,2).is_left_of(Location(2,2)))  # True
#for n in Location(0,0).neighbours():
#  print(n)
#print(Location(0,0).neighbours())               # should be [(1,0), (0,1)] in some order
#loc = Location(3,0)
#print(loc.forward(Orientation.E), loc)          # True (bump), (3,0) unchanged
#loc2 = Location(1,1)
#print(loc2.forward(Orientation.N), loc2)        # False, now (1,2)

In [41]:
# Let's test Location class
# print(Location.random())
# print(Location.from_linear(15))
# print(Location(0,1).to_linear())

In [42]:
class Environment():
  wumpus_location: Location
  wumpus_alive: bool
  agent_location: Location
  agent_orientation: Orientation
  agent_has_arrow: bool
  agent_has_gold: bool
  game_over: bool
  gold_location: Location
  pit_locations: List[Location]
  time_step: int
  pit_prob: float
  allow_climb_without_gold: bool

  def init(self, pit_prob: float, allow_climb_without_gold: bool):
    self.allow_climb_without_gold=allow_climb_without_gold
    self.pit_prob=pit_prob
    self.agent_location=Location(0,0)
    self.agent_orientation=Orientation.E
    self.agent_has_arrow=True
    self.agent_has_gold=False
    self.game_over=False
    self.time_step=0

    self.make_wumpus()
    self.make_gold()
    self.make_pits(pit_prob)

    return Percept(time_step=0,
                   bump= False,
                   breeze=self.is_breeze(),
                   stench=self.is_stench(),
                   scream=False,
                   glitter=self.is_glitter(),
                   reward=0,
                   done=self.game_over)

  def make_wumpus(self):
    self.wumpus_alive = True
    while True:
      loc = Location.random()
      if not loc.is_location(Location(0,0)):
        self.wumpus_location = loc
        return

  def make_gold(self):
    while True:
      loc = Location.random()
      if not loc.is_location(Location(0,0)):
        self.gold_location = loc
        return

  def make_pits(self, pit_prob):
    self.pit_locations = []
    for n in range(16):
      loc = Location.from_linear(n)
      if loc.is_location(Location(0,0)):
        continue
      if random.random() < pit_prob:
        self.pit_locations.append(loc)
    return

  def is_pit_at(self, location)->bool:
    for p in self.pit_locations:
      if p.is_location(location):
        return True
    return False

  def is_wumpus_at(self, location)->bool:
    return self.wumpus_location.is_location(location)

  def is_agent_at(self, location)->bool:
    return self.agent_location.is_location(location)

  def is_gold_at(self, location)->bool:
    if self.agent_has_gold:
      return self.agent_location.is_location(location)
    return self.gold_location.is_location(location)

  def is_pit_adjacent_to_agent(self)->bool:
    neighbours = self.agent_location.neighbours()
    for p in self.pit_locations:
      for n in neighbours:
        if p.is_location(n):
          return True
    return False

  def is_wumpus_adjacent_to_agent(self)->bool:
    neighbours = self.agent_location.neighbours()
    for n in neighbours:
      if self.wumpus_location.is_location(n):
        return True
    return False

  def is_agent_at_hazard(self)->bool:
        return any(self.agent_location.is_location(p) for p in self.pit_locations) or (self.wumpus_location.is_location(self.agent_location) and self.wumpus_alive)

  def is_glitter(self)->bool:
        return self.is_gold_at(self.agent_location)

  def is_breeze(self)->bool:
        return self.is_pit_adjacent_to_agent() or any(p.is_location(self.agent_location) for p in self.pit_locations)

  def is_stench(self)->bool:
        return self.is_wumpus_adjacent_to_agent() or self.agent_location.is_location(self.wumpus_location)


  def wumpus_in_line_of_fire(self)->bool:
    # return true if the wumpus is a cell the arrow would pass through if fired
    ax, ay = self.agent_location.x, self.agent_location.y
    wx, wy = self.wumpus_location.x, self.wumpus_location.y

    if self.agent_orientation == Orientation.E:
      return wx > ax and wy == ay

    if self.agent_orientation == Orientation.S:
      return wx == ax and wy < ay

    if self.agent_orientation == Orientation.W:
      return wx < ax and wy == ay

    return wx == ax and wy > ay

  def kill_attempt(self)->bool:
    if self.wumpus_in_line_of_fire() and self.wumpus_alive:
      self.wumpus_alive = False
      return True
    return False

  def step(self, action: Action) -> Percept:
    if self.game_over:
        return Percept(
            time_step=self.time_step,
            bump=False,
            breeze=self.is_breeze(),
            stench=self.is_stench(),
            scream=False,
            glitter=self.is_glitter(),
            reward=0,
            done=True
        )

    self.time_step += 1
    reward = -1
    bump = False
    scream = False

    if action == Action.LEFT:
        self.agent_orientation = self.agent_orientation.turn_left()

    elif action == Action.RIGHT:
        self.agent_orientation = self.agent_orientation.turn_right()

    elif action == Action.FORWARD:
        bump = self.agent_location.forward(self.agent_orientation)

    elif action == Action.GRAB:
        if self.agent_location.is_location(self.gold_location):
            self.agent_has_gold = True

    elif action == Action.SHOOT:
        if self.agent_has_arrow:
            self.agent_has_arrow = False
            reward -= 10
            scream = self.kill_attempt()

    elif action == Action.CLIMB:
        at_start = self.agent_location.is_location(Location(0, 0))
        if at_start:
            if self.agent_has_gold:
                reward += 1000
                self.game_over = True
            elif self.allow_climb_without_gold:
                self.game_over = True

    if not self.game_over and self.is_agent_at_hazard():
        reward -= 1000
        self.game_over = True

    return Percept(
        time_step=self.time_step,
        bump=bump,
        breeze=self.is_breeze(),
        stench=self.is_stench(),
        scream=scream,
        glitter=self.is_glitter(),
        reward=reward,
        done=self.game_over
    )
  # Visualize the game state
  def visualize(self):
      for y in range(3, -1, -1):
          line = '|'
          for x in range(0, 4):
              loc = Location(x, y)
              cell_symbols = [' ', ' ', ' ', ' ']
              if self.is_agent_at(loc): cell_symbols[0] = self.agent_orientation.symbol()
              if self.is_pit_at(loc): cell_symbols[1] = 'P'
              if self.is_wumpus_at(loc):
                  if self.wumpus_alive:
                      cell_symbols[2] = 'W'
                  else:
                      cell_symbols[2] = 'w'
              if self.is_gold_at(loc): cell_symbols[3] = 'G'
              for char in cell_symbols: line += char
              line += '|'
          print(line)

In [43]:
# Let's test the Environment class
#nv = Environment()
#p = env.init(0.0, True)
#env.visualize()
#print(p)
#p = env.step(Action.SHOOT)
#print(p.reward)
#p2 = env.step(Action.SHOOT)
#print(p2.reward)
#env.agent_location = Location(0,0)
#env.agent_orientation = Orientation.W
#p = env.step(Action.FORWARD)
#print(p.bump, env.agent_location)
#env = Environment()
#env.init(0.0, True)
#env.agent_location = Location(1,0)
#p = env.step(Action.CLIMB)
#print(p.done)  # should be False (not at start)


Let's start with assignment 2.

In [44]:
class MovePlanningAgent():
  def __init__(self):
    self.has_gold=False
    self.loc = Location(0,0)
    self.ori = Orientation.E
    self.visited = set()
    self.graph = nx.DiGraph() #Directed Graph, meaning direction matters
    self.plan = [] #List of planned actions to execute after grabbing the gold.

  #Helper representing graph nodes(states)
  def state(self, x, y, ori):
    return (x,y,ori)

  def add_cell(self,x,y):
    #1) add the 4 orientation nodes
    for ori in [Orientation.N, Orientation.E, Orientation.S, Orientation.W]:
      self.graph.add_node(self.state(x,y,ori))
    #2) add turning edges (stay in same cell)
    for ori in [Orientation.N, Orientation.E, Orientation.S, Orientation.W]:
      self.graph.add_edge(self.state(x,y,ori),self.state(x,y,ori.turn_left()),action=Action.LEFT)
      self.graph.add_edge(self.state(x,y,ori),self.state(x,y,ori.turn_right()),action=Action.RIGHT)

  #Add a helper to compute the forward neighbor
  def forward_cell(self, x, y, ori):
    if ori == Orientation.N:
        return (x, y+1)
    elif ori == Orientation.E:
        return (x+1, y)
    elif ori == Orientation.S:
        return (x, y-1)
    elif ori == Orientation.W:
        return (x-1, y)

  #Add a function that adds forward edges for one cell
  def add_forward_edge(self,x,y,ori):
    nx,ny= self.forward_cell(x,y,ori) #returns a tuple
    if (nx,ny) in self.visited:
        self.graph.add_edge(self.state(x,y,ori),self.state(nx,ny,ori),action=Action.FORWARD)

  def add_forward_edges(self,x,y):
    for ori in [Orientation.N, Orientation.E, Orientation.S, Orientation.W]:
      self.add_forward_edge(x,y,ori)

  def choose_action(self, percept: Percept):
    #if we already have a plan we don't go with random
    if self.plan:
      return self.plan.pop(0)

    #if we have a gold and we are at the goal location we just need to climb
    if self.has_gold and (self.loc.x, self.loc.y)==(0,0):
      return Action.CLIMB

    #if percept is glitter -> grab
    if percept.glitter and not self.has_gold:
      self.has_gold=True
      return Action.GRAB

    #if self.plan is empty, we are not in the start and we already grabbed the gold, we need to fill in the self.plan
    if self.has_gold and not self.plan and (self.loc.x, self.loc.y)!=(0,0):
      self.plan = self.bfs_actions_to_start()
      print("PLANNING RETURN:", self.plan)
      if self.plan:
        return self.plan.pop(0)

    #ow choose another action
    return random.choice([Action.LEFT, Action.RIGHT, Action.FORWARD])

  def update_state(self, action, new_percept):
    if action == Action.LEFT:
        self.ori = self.ori.turn_left()

    elif action == Action.RIGHT:
        self.ori = self.ori.turn_right()

    elif action == Action.FORWARD:
        if not new_percept.bump:
          if self.ori == Orientation.E:
            self.loc = Location(self.loc.x+1,self.loc.y)

          elif self.ori == Orientation.S:
            self.loc = Location(self.loc.x, self.loc.y-1)

          elif self.ori == Orientation.W:
            self.loc = Location(self.loc.x-1, self.loc.y)

          elif self.ori == Orientation.N:
            self.loc = Location(self.loc.x, self.loc.y+1)

  def bfs_actions_to_start(self):
    start_state = self.state(self.loc.x,self.loc.y,self.ori)

    goals = {
        (0,0,Orientation.N),
        (0,0,Orientation.E),
        (0,0,Orientation.S),
        (0,0,Orientation.W)
    }

    q = deque([start_state])

    parent = {start_state:None}
    parent_action = {start_state:None}

    while q:
      s = q.popleft()

      if s in goals:
        actions = []

        while parent[s] is not None:
          actions.append(parent_action[s])
          s=parent[s]

        actions.reverse()
        return actions

      for nxt in self.graph.successors(s):
        if nxt not in parent:
          parent[nxt] = s
          parent_action[nxt]= self.graph[s][nxt]["action"]
          q.append(nxt)

    return []

  def run(self):
    env = Environment() # this is creating an object of env from Environment Class
    cumulative_reward = 0
    percept = env.init(0.2, False)
    #percept = env.init(0.0, False)
    self.visited.add((self.loc.x,self.loc.y))
    self.add_cell(self.loc.x,self.loc.y)
    self.add_forward_edges(self.loc.x,self.loc.y)

    while not percept.done:
      env.visualize()
      print("Percept:", percept)
      print("Has Gold:", self.has_gold)
      action = self.choose_action(percept)
      print("Action:", action)
      old = (self.loc.x,self.loc.y)
      percept = env.step(action)
      self.update_state(action,percept)
      new = (self.loc.x,self.loc.y)
      print(new,self.ori,percept.bump)
      entered_new_cell = (old !=new ) and (not percept.done)
      if entered_new_cell:
        self.visited.add(new)
        self.add_cell(new[0],new[1])
        self.add_forward_edges(new[0],new[1])
        x, y= new
        for (vx,vy) in self.visited:
          if abs(vx-x) + abs(vy-y) == 1:
            self.add_forward_edges(vx,vy)
      cumulative_reward += percept.reward

    env.visualize()
    print("Visited count:", len(self.visited))
    print("Graph nodes:", self.graph.number_of_nodes())
    print("Graph edges:", self.graph.number_of_edges())
    print("Percept:", percept)
    print('Cumulative reward:', cumulative_reward)

In [45]:
#let's run some tests
#agent = MovePlanningAgent()
#agent.run()

Let's start with assignment 3.

In [46]:
class ProbAgent(MovePlanningAgent):
    def __init__(self):
        super().__init__()

        # Breeze memory
        self.breeze_cells = set()
        self.no_breeze_cells = set()

        # Stench memory
        self.stench_cells = set()
        self.no_stench_cells = set()

        # Scream / Wumpus state
        self.heard_scream = False
        self.wumpus_dead = False

        # Arrow state
        self.arrow_used = False

        # Exploration target/path memory
        self.current_target = None
        self.target_path = []

        # Deterministic knowledge
        self.known_no_pit = set()
        self.known_no_wumpus = set()

    def update_beliefs(self, percept: Percept):
        current = (self.loc.x, self.loc.y)

        # Mark current location as visited
        self.visited.add(current)

        # Record breeze evidence
        if percept.breeze:
            self.breeze_cells.add(current)
            self.no_breeze_cells.discard(current)
        else:
            self.no_breeze_cells.add(current)
            self.breeze_cells.discard(current)

        # Record stench evidence
        if percept.stench:
            self.stench_cells.add(current)
            self.no_stench_cells.discard(current)
        else:
            self.no_stench_cells.add(current)
            self.stench_cells.discard(current)

        # Record scream evidence
        if percept.scream:
            self.heard_scream = True
            self.wumpus_dead = True

    def choose_action(self, percept):
        self.update_beliefs(percept)
        self.update_deterministic_knowledge()
        return super().choose_action(percept)

    # Part 2: Frontier detection
    def get_neighbors(self, x, y):
        candidates = [
            (x + 1, y),
            (x, y + 1),
            (x - 1, y),
            (x, y - 1)
        ]
        return [(x, y) for (x, y) in candidates if 0 <= x < 4 and 0 <= y < 4]

    def get_frontier_cells(self):
        frontier = set()

        for (x, y) in self.visited:
            for neighbor in self.get_neighbors(x, y):
                if neighbor not in self.visited:
                    frontier.add(neighbor)

        return frontier

    # Part 3: Logic before probability
    def update_deterministic_knowledge(self):
        # Any visited cell is safe from pit and Wumpus
        for cell in self.visited:
            self.known_no_pit.add(cell)
            self.known_no_wumpus.add(cell)

        # If a visited cell had no breeze, all neighbors are pit-free
        for (x, y) in self.no_breeze_cells:
            for neighbor in self.get_neighbors(x, y):
                self.known_no_pit.add(neighbor)

        # If a visited cell had no stench, all neighbors are Wumpus-free
        for (x, y) in self.no_stench_cells:
            for neighbor in self.get_neighbors(x, y):
                self.known_no_wumpus.add(neighbor)

        # If scream was heard, Wumpus is dead everywhere
        if self.heard_scream:
            for x in range(4):
                for y in range(4):
                    self.known_no_wumpus.add((x, y))

    def get_known_safe_frontier(self):
        frontier = self.get_frontier_cells()
        safe_frontier = set()

        for cell in frontier:
            if cell in self.known_no_pit and cell in self.known_no_wumpus:
                safe_frontier.add(cell)

        return safe_frontier

    # Part 4: Probabilistic pit model
    def all_cells(self):
        return [(x, y) for x in range(4) for y in range(4)]

    def pit_var_name(self, cell):
        x, y = cell
        return f"Pit_{x}_{y}"

    def breeze_var_name(self, cell):
        x, y = cell
        return f"Breeze_{x}_{y}"

    # Step 4.4.1: Breeze CPT rows
    def build_breeze_cpt_rows(self, neighbors):
        rows = []

        for values in product([True, False], repeat=len(neighbors)):
            breeze = any(values)

            rows.append(list(values) + [True, 1.0 if breeze else 0.0])
            rows.append(list(values) + [False, 1.0 if not breeze else 0.0])

        return rows

    # Step 4.4.2: Pit prior
    def build_pit_prior(self, cell):
        if cell == (0, 0):
            return Categorical([[0.0, 1.0]])
        return Categorical([[0.2, 0.8]])

    def create_pit_distributions(self):
        pit_dists = {}

        for cell in self.all_cells():
            pit_dists[cell] = self.build_pit_prior(cell)

        return pit_dists

    def create_breeze_distributions(self, pit_dists):
        breeze_dists = {}

        for cell in self.all_cells():
            x, y = cell
            neighbors = self.get_neighbors(x, y)
            cpt_rows = self.build_breeze_cpt_rows(neighbors)
            parent_dists = [pit_dists[n] for n in neighbors]

            breeze_dists[cell] = ConditionalCategorical(cpt_rows, parent_dists)

        return breeze_dists

    # Steps 4.4.3 / 4.4.4 / 4.4.5
    def create_pit_bayesian_network(self):
        pit_dists = self.create_pit_distributions()
        breeze_dists = self.create_breeze_distributions(pit_dists)

        distributions = []

        # Pit nodes first
        for cell in self.all_cells():
            distributions.append(pit_dists[cell])

        # Breeze nodes second
        for cell in self.all_cells():
            distributions.append(breeze_dists[cell])

        edges = []

        for cell in self.all_cells():
            x, y = cell
            neighbors = self.get_neighbors(x, y)

            for n in neighbors:
                edges.append((pit_dists[n], breeze_dists[cell]))

        model = BayesianNetwork(distributions=distributions, edges=edges)
        return model

    # Step 4.5: Build evidence vector
    def build_pit_evidence(self):
        evidence = []

        # Pit nodes
        for cell in self.all_cells():
            if cell in self.known_no_pit:
                evidence.append(False)
            elif cell in self.visited:
                evidence.append(False)
            else:
                evidence.append(None)

        # Breeze nodes
        for cell in self.all_cells():
            if cell in self.breeze_cells:
                evidence.append(True)
            elif cell in self.no_breeze_cells:
                evidence.append(False)
            else:
                evidence.append(None)

        return evidence

    # Step 4.6: Query P(pit at cell)
    def get_pit_probability(self, cell):
        # Deterministic shortcuts
        if cell in self.known_no_pit:
            return 0.0
        elif cell in self.visited:
            return 0.0

        # Build model
        model = self.create_pit_bayesian_network()

        # Build evidence
        evidence = self.build_pit_evidence()

        # Run inference
        beliefs = model.predict_proba([evidence])[0]

        # Find pit node index
        pit_index = self.all_cells().index(cell)

        # Get posterior distribution
        dist = beliefs[pit_index]

        # Return P(pit=True)
        return dist.parameters[0][0]

    #Build the Wumpus probabilistic model
    def build_stench_cpt_rows(self, neighbors):
      rows = []

        for values in product([True, False], repeat=len(neighbors)):
            stench = any(values)

            rows.append(list(values) + [True, 1.0 if stench else 0.0])
            rows.append(list(values) + [False, 1.0 if not stench else 0.0])

        return rows

    def build_wumpus_prior(self, cell): # check this out
        if cell == (0, 0):
            return Categorical([[0.0, 1.0]])
        return Categorical([[0.2, 0.8]])

    def create_wumpus_distributions(self):
      wumpus_dists = {}

        for cell in self.all_cells():
            wumpus_dists[cell] = self.build_wumpus_prior(cell)

        return wumpus_dists

    def create_stench_distributions(self, wumpus_dists):
      stench_dists = {}

        for cell in self.all_cells():
            x, y = cell
            neighbors = self.get_neighbors(x, y)
            cpt_rows = self.build_stench_cpt_rows(neighbors)
            parent_dists = [wumpus_dists[n] for n in neighbors]

            stench_dists[cell] = ConditionalCategorical(cpt_rows, parent_dists)

        return stench_dists

    def create_wumpus_bayesian_network(self):





build_wumpus_evidence(self)
get_wumpus_probability(self, cell)




SyntaxError: invalid syntax (621208438.py, line 242)